# Tennis Betting Pipeline – End-to-End Demo

This notebook demonstrates the upgraded decision pipeline on top of the existing NN model.

## Architecture

```
NN probability
    ↓
Calibration          → aligns raw sigmoid to observed frequencies
    ↓
MC Dropout           → confidence_multiplier (how certain is the model?)
    ↓
OOD Detection        → ood_shrinkage + hierarchical prior blend
    ↓
Edge Estimation      → usable_edge = p_calibrated − q_fair − λ·uncertainty
    ↓
Sizing Engine        → stake = fractional_Kelly × caps
```

## Key principle

> Do **not** size directly off raw NN probability.  
> Size off: **calibrated probability − market price − uncertainty penalty**.

## 0. Setup – run data prep + model training first

The cells below assume you have already executed `Untitled2.ipynb` through cell 22
so that `model_embed`, `X_train`, `X_test`, `y_train`, `y_test`, `A_train`, `A_test`,
`B_train`, `B_test`, `scaler`, and `imputer` are in memory.

If running standalone, execute the training notebook first.

In [ ]:
import sys
import numpy as np
import pandas as pd

# Make sure src/ is importable
if '.' not in sys.path:
    sys.path.insert(0, '.')

from src.calibration   import ProbabilityCalibrator
from src.uncertainty   import MCDropoutEstimator
from src.ood_detector  import MahalanobisOOD, HierarchicalPrior
from src.edge_estimator import EdgeEstimator
from src.sizing_engine  import SizingEngine, BettingPolicy
from src.pipeline       import TennisBettingPipeline, PipelineConfig

print('✅ All pipeline modules imported successfully')

---
## 1. Layer 1 – Probability Calibration

The NN sigmoid is not guaranteed to be calibrated.  
We fit isotonic regression on a held-out calibration slice.

In [ ]:
# ── Split test set into calibration set + final evaluation set ──────────
# We use the first 50 % of the test set to fit the calibrator.
# The second 50 % is untouched for final evaluation.

n_test = len(X_test)
n_calib = n_test // 2

X_calib,  X_eval  = X_test[:n_calib],  X_test[n_calib:]
A_calib,  A_eval  = A_test[:n_calib],  A_test[n_calib:]
B_calib,  B_eval  = B_test[:n_calib],  B_test[n_calib:]
y_calib,  y_eval  = y_test[:n_calib],  y_test[n_calib:]

# Raw NN probabilities on calibration set
raw_calib_probs = model_embed.predict(
    [X_calib, A_calib, B_calib], verbose=0
).ravel()

# Fit calibrator
calibrator = ProbabilityCalibrator(method='isotonic')
calibrator.fit(raw_calib_probs, y_calib)

# Apply to eval set
raw_eval_probs = model_embed.predict(
    [X_eval, A_eval, B_eval], verbose=0
).ravel()
calib_eval_probs = calibrator.transform(raw_eval_probs)

# Measure improvement
metrics = calibrator.calibration_metrics(raw_eval_probs, calib_eval_probs, y_eval)
print('Calibration improvement:')
print(f"  ECE before:   {metrics['ece_before']:.4f}")
print(f"  ECE after:    {metrics['ece_after']:.4f}")
print(f"  Brier before: {metrics['brier_before']:.4f}")
print(f"  Brier after:  {metrics['brier_after']:.4f}")

In [ ]:
# Reliability diagram
calibrator.plot_reliability_diagram(raw_eval_probs, calib_eval_probs, y_eval)

---
## 2. Layer 2 – MC Dropout Uncertainty Estimation

Run N stochastic forward passes (dropout ON) to get mean + std per match.

In [ ]:
mc_estimator = MCDropoutEstimator(model_embed, n_samples=50)  # 50 for speed; use 100+ in production

# Run on a small sample for demo speed
demo_n = 500
mc_inputs = [X_eval[:demo_n], A_eval[:demo_n], B_eval[:demo_n]]

mc_means, mc_stds = mc_estimator.predict(mc_inputs, has_player_ids=True)

print(f'Mean uncertainty (std): {mc_stds.mean():.4f}')
print(f'90th pct uncertainty:   {np.percentile(mc_stds, 90):.4f}')
print(f'Matches with std > 0.10: {(mc_stds > 0.10).mean():.1%}')

conf_multipliers = mc_estimator.confidence_multiplier(mc_stds)
print(f'\nMean confidence multiplier: {conf_multipliers.mean():.3f}')

# Distribution plot
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(mc_stds, bins=40, edgecolor='k')
axes[0].set_xlabel('MC Dropout std')
axes[0].set_ylabel('Count')
axes[0].set_title('Uncertainty Distribution')

axes[1].hist(conf_multipliers, bins=40, edgecolor='k', color='seagreen')
axes[1].set_xlabel('Confidence Multiplier')
axes[1].set_title('Confidence Multiplier Distribution')
plt.tight_layout()
plt.show()

---
## 3. Layer 3 – OOD Detection

Flag matches whose feature vectors are far from the training distribution.

In [ ]:
ood_detector = MahalanobisOOD(threshold_pct=95, shrink_min=0.0)
ood_detector.fit(X_train)

ood_scores   = ood_detector.score(X_eval[:demo_n])
ood_flags    = ood_detector.is_ood(X_eval[:demo_n])
ood_shrinkages = ood_detector.shrinkage(X_eval[:demo_n])

print(f'OOD rate (eval set): {ood_flags.mean():.1%}')
print(f'Mean OOD shrinkage:  {ood_shrinkages.mean():.3f}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(ood_scores, bins=40, edgecolor='k')
axes[0].axvline(ood_detector._threshold, color='red', linestyle='--', label='OOD threshold')
axes[0].set_xlabel('Mahalanobis distance')
axes[0].set_title('OOD Score Distribution')
axes[0].legend()

axes[1].hist(ood_shrinkages, bins=40, edgecolor='k', color='tomato')
axes[1].set_xlabel('OOD Shrinkage Factor')
axes[1].set_title('OOD Shrinkage (1=in-dist, 0=skip)')
plt.tight_layout()
plt.show()

---
## 4. Layer 4 – Edge Estimation

Simulate market odds and compute usable edge for each match in the eval set.

In production you would feed real live odds.  Here we simulate:
- Market odds derived from NN calibrated probability ± a small book margin.

In [ ]:
np.random.seed(42)

# Simulate market: book sets odds based on true probability + 5% margin
# True implied prob for A is calib_eval_probs[:demo_n]
true_p = calib_eval_probs[:demo_n]

# Add a small random market bias (market sometimes misprices)
market_p = np.clip(true_p + np.random.normal(0, 0.04, demo_n), 0.05, 0.95)

# Add 4% overround across both sides
overround = 1.04
decimal_odds_A = overround / market_p            # odds for A
decimal_odds_B = overround / (1 - market_p)      # odds for B

edge_estimator = EdgeEstimator(
    uncertainty_lambda=0.5,
    min_edge=0.03,
    fee_rate=0.0,
)

edge_results = edge_estimator.estimate_batch(
    p_calibrated=true_p,
    decimal_odds_A=decimal_odds_A,
    decimal_odds_B=decimal_odds_B,
    uncertainty=mc_stds,
)

summary = edge_estimator.edge_summary(edge_results)
print('Edge summary:')
for k, v in summary.items():
    print(f'  {k}: {v:.4f}' if isinstance(v, float) else f'  {k}: {v}')

usable_edges = np.array([r.usable_edge for r in edge_results])
plt.figure(figsize=(8, 4))
plt.hist(usable_edges, bins=50, edgecolor='k')
plt.axvline(0.03, color='red', linestyle='--', label='Min edge (3%)')
plt.xlabel('Usable Edge')
plt.ylabel('Count')
plt.title('Usable Edge Distribution')
plt.legend()
plt.show()

---
## 5. Layer 5 – Sizing Engine

Convert usable edges to concrete stake fractions using fractional Kelly with hard caps.

In [ ]:
policy = BettingPolicy(
    kelly_coeff=0.25,              # quarter Kelly
    max_per_bet_fraction=0.010,    # cap at 1 % bankroll per bet
    max_per_match_fraction=0.030,  # cap at 3 % across one match
    max_daily_loss_fraction=0.050, # halt at -5 % daily drawdown
    min_edge=0.03,
    min_confidence_multiplier=0.30,
    min_ood_shrinkage=0.20,
)

engine = SizingEngine(policy=policy)

sizing_decisions = engine.size_batch(
    usable_edges=usable_edges,
    decimal_odds=decimal_odds_A,
    confidence_multipliers=conf_multipliers,
    ood_shrinkages=ood_shrinkages,
    bankroll=10_000.0,
)

summary = engine.decision_summary(sizing_decisions)
print('Sizing summary:')
for k, v in summary.items():
    if k == 'reasons':
        print(f'  {k}: {v}')
    else:
        print(f'  {k}: {v:.4f}' if isinstance(v, float) else f'  {k}: {v}')

# Plot stake distribution for bets that were placed
stakes = [d.stake_fraction * 100 for d in sizing_decisions if d.action == 'BET']
if stakes:
    plt.figure(figsize=(8, 4))
    plt.hist(stakes, bins=30, edgecolor='k', color='steelblue')
    plt.xlabel('Stake (% of bankroll)')
    plt.ylabel('Count')
    plt.title(f'Stake Distribution ({len(stakes)} bets)')
    plt.show()

---
## 6. Full Pipeline – One-Call Interface

Using `TennisBettingPipeline` to unify all layers.

In [ ]:
config = PipelineConfig(
    calibration_method='isotonic',
    mc_samples=50,
    uncertainty_lambda=0.5,
    min_edge=0.03,
    kelly_coeff=0.25,
    max_per_bet_fraction=0.010,
)

pipeline = TennisBettingPipeline(model_embed, config=config, has_player_ids=True)

# Offline fit (slow layer)
pipeline.fit(
    X_train=X_train,
    y_train=y_train,
    raw_probs_calib=raw_calib_probs,
    y_calib=y_calib,
)
print('✅ Pipeline fitted')

In [ ]:
# Single-match live decision
match_idx = 0  # use first eval match as example

decision = pipeline.decide(
    X_live=X_eval[match_idx:match_idx+1],
    decimal_odds_A=float(decimal_odds_A[match_idx]),
    decimal_odds_B=float(decimal_odds_B[match_idx]),
    player_A_id=int(A_eval[match_idx]),
    player_B_id=int(B_eval[match_idx]),
    bankroll=10_000.0,
    match_meta={'match_idx': match_idx},
)

print('=== Pipeline Decision ===')
print(f'Raw NN probability:       {decision.raw_prob:.4f}')
print(f'Calibrated probability:   {decision.calibrated_prob:.4f}')
print(f'MC Dropout std:           {decision.mc_std:.4f}')
print(f'Confidence multiplier:    {decision.confidence_multiplier:.4f}')
print(f'OOD score:                {decision.ood_score:.4f}')
print(f'OOD shrinkage:            {decision.ood_shrinkage:.4f}')
print(f'Is OOD:                   {decision.is_ood}')
print(f'Blended probability:      {decision.blended_prob:.4f}')
print()
print(f'Market implied prob (A):  {decision.edge.q_fair:.4f}')
print(f'Raw edge:                 {decision.edge.raw_edge:.4f}')
print(f'Uncertainty penalty:      {decision.edge.uncertainty_penalty:.4f}')
print(f'Usable edge:              {decision.edge.usable_edge:.4f}')
print()
print(f'ACTION:                   {decision.sizing.action}')
print(f'Reason:                   {decision.sizing.reason}')
print(f'Stake fraction:           {decision.sizing.stake_fraction:.4f}  ({decision.sizing.stake_fraction*100:.2f}%)')
print(f'Stake dollars ($10k bank):{decision.sizing.stake_dollars:.2f}')

---
## 7. Backtest – Full Eval Set

In [ ]:
n_backtest = demo_n  # use first 500 eval matches for speed

all_decisions = pipeline.decide_batch(
    X_batch=X_eval[:n_backtest],
    decimal_odds_A=decimal_odds_A[:n_backtest],
    decimal_odds_B=decimal_odds_B[:n_backtest],
    A_ids=A_eval[:n_backtest],
    B_ids=B_eval[:n_backtest],
    bankroll=10_000.0,
)

log = TennisBettingPipeline.log_decisions(all_decisions, outcomes=y_eval[:n_backtest])

print('=== Backtest Summary ===')
for k, v in log.items():
    print(f'  {k}: {v:.4f}' if isinstance(v, float) else f'  {k}: {v}')

In [ ]:
# Calibration-by-bucket: predicted vs actual win rate per edge bucket
edge_vals = np.array([d.edge.usable_edge for d in all_decisions])
calib_vals = np.array([d.calibrated_prob for d in all_decisions])
outcomes_arr = y_eval[:n_backtest]

# Bucket by calibrated probability
bins = np.linspace(0.3, 0.7, 8)
bucket_idx = np.digitize(calib_vals, bins)
bucket_stats = []
for b in range(1, len(bins)):
    mask = bucket_idx == b
    if mask.sum() > 5:
        bucket_stats.append({
            'pred_center': (bins[b-1] + bins[b]) / 2,
            'actual_win_rate': outcomes_arr[mask].mean(),
            'count': mask.sum(),
        })

if bucket_stats:
    df_buckets = pd.DataFrame(bucket_stats)
    plt.figure(figsize=(8, 5))
    plt.plot([0.3, 0.7], [0.3, 0.7], 'k--', label='Perfect')
    plt.plot(df_buckets['pred_center'], df_buckets['actual_win_rate'], 'o-', label='Model')
    plt.xlabel('Calibrated Predicted Probability')
    plt.ylabel('Actual Win Rate')
    plt.title('Calibration by Probability Bucket')
    plt.legend()
    plt.grid(True)
    plt.show()

---
## 8. Rare-State Decision Tree

For each incoming match, the pipeline automatically routes through:

```
Is this state common enough?
├── YES (low OOD score)     → use normal model + full sizing
├── SOMEWHAT RARE           → model + OOD shrinkage + reduced size
└── VERY RARE / NEW         → hierarchical prior blend + tiny size or skip
```

This is handled automatically by `ood_detector.shrinkage()` + `HierarchicalPrior.blend()`.
No special case coding required.

In [ ]:
# Demonstrate OOD routing on synthetic extreme cases
from src.ood_detector import HierarchicalPrior

prior = HierarchicalPrior()
prior.fit(
    surface_priors={'Clay': 0.51, 'Grass': 0.49, 'Hard_outdoor': 0.50, 'Hard_indoor': 0.50},
    overall_prior=0.50,
)

scenarios = [
    ('In-distribution',   0.68, 0.00, 'Clay',   None),
    ('Mildly OOD',        0.68, 0.40, 'Clay',   None),
    ('Very OOD (surface prior)', 0.68, 0.80, 'Clay', None),
    ('Very OOD (market prior)',  0.68, 0.80, None,   0.58),
    ('Maximally OOD',     0.68, 1.00, None,   None),
]

print(f'{"Scenario":<35} {"p_model":>8} {"ood":>6} {"blended":>8}')
print('-' * 60)
for name, p_model, ood_score, surface, mkt in scenarios:
    blended = prior.blend(p_model, ood_score, surface=surface, market_price=mkt)
    print(f'{name:<35} {p_model:>8.3f} {ood_score:>6.2f} {blended:>8.4f}')

---
## 9. Policy Sensitivity Analysis

Show how different Kelly fractions affect total exposure and bet frequency.

In [ ]:
kelly_coeffs = [0.10, 0.25, 0.50]
rows = []
for kc in kelly_coeffs:
    p = BettingPolicy(kelly_coeff=kc, max_per_bet_fraction=0.01, min_edge=0.03)
    eng = SizingEngine(policy=p)
    decs = eng.size_batch(
        usable_edges=usable_edges,
        decimal_odds=decimal_odds_A,
        confidence_multipliers=conf_multipliers,
        ood_shrinkages=ood_shrinkages,
        bankroll=10_000.0,
    )
    s = eng.decision_summary(decs)
    rows.append({
        'Kelly coeff': kc,
        'Bet rate': f"{s['bet_rate']:.1%}",
        'Mean stake %': f"{s['mean_stake_pct']:.3f}%",
        'Total exposure %': f"{s['total_exposure_pct']:.2f}%",
    })

print(pd.DataFrame(rows).to_string(index=False))

---
## Summary

The upgraded pipeline implements the full decision stack:

| Module | What it does |
|--------|-------------|
| `ProbabilityCalibrator` | Fixes NN overconfidence via isotonic/Platt regression |
| `MCDropoutEstimator` | Measures model uncertainty per match via stochastic forward passes |
| `MahalanobisOOD` | Detects rare/unseen feature combinations |
| `HierarchicalPrior` | Gracefully degrades to surface/market priors when OOD |
| `EdgeEstimator` | `usable_edge = p_calibrated − q_fair − λ·uncertainty` |
| `SizingEngine` | Fractional Kelly (0.25×) with hard caps on exposure |
| `TennisBettingPipeline` | Orchestrates all layers; event-driven, not constant recomputation |

**Key rules enforced automatically:**
- No bet if `usable_edge < 3%`
- No bet if uncertainty is very high (`std > high_std` threshold)
- No bet if state is too OOD (`shrinkage < 20%`)
- Max 1% bankroll per bet (hard cap)
- Max 3% per match across all positions
- Daily stop-loss at -5%